# Module 5 (Bonus) · Building a Data App with Streamlit
### FastAPI for ML Engineers — Bootcamp Bonus Session

**Time budget:** ~45–60 minutes
**Prereq:** Modules 1–2 (works standalone too — this module retrains its own tiny model)

## Agenda
| Time | Topic |
|---|---|
| 0:00–0:10 | What Streamlit is, and where it fits next to FastAPI |
| 0:10–0:30 | Building a data app: widgets, layout, single + batch prediction |
| 0:30–0:40 | Caching the model with `@st.cache_resource` |
| 0:40–0:50 | Pattern 2: Streamlit as a thin frontend calling your FastAPI backend |
| 0:50–1:00 | Dockerizing it + exercise |

## Learning objectives
- Explain what Streamlit is for, and — just as importantly — what it's *not* for
- Build an interactive data app around a trained model: inputs, single prediction, batch CSV scoring, charts
- Cache expensive resources (model loads) correctly
- Know the two architecture patterns: model-in-process vs. calling your FastAPI service, and when to pick each
- Containerize the app the same way you containerized FastAPI in Module 3

## 5.1 What is Streamlit, and where does it fit?

**Streamlit** turns a plain Python script into an interactive web app — no HTML/CSS/JS required. You write top-to-bottom Python; Streamlit re-runs the whole script on every interaction (a button click, a slider drag) and redraws the UI.

### Where it fits next to what you already built
FastAPI (Modules 1–4) is a **backend**: fast, built for machine-to-machine traffic, no UI of its own. Streamlit is a **frontend/tooling layer**: built for humans to poke at something — a model, a dataset, a pipeline — without you hand-writing a React app.

| | FastAPI | Streamlit |
|---|---|---|
| Built for | Programmatic clients (services, apps, curl) | Humans clicking around in a browser |
| Typical use in ML | Production model serving | Internal demos, QA tools, stakeholder-facing exploration, rapid prototyping |
| Scales to high QPS? | Yes, that's the point | Not really the point — one user's session runs the whole script |
| Where it runs | Anywhere (Docker, Vertex AI, Cloud Run, GKE) | Anywhere Docker/Cloud Run/GKE runs a web process; **not** a Vertex AI Model/Endpoint — that resource type is specifically for the prediction-container contract from Module 4, not general web apps |

A very common real pattern: **Streamlit app → calls your FastAPI service → which calls the model.** That's the separation we'll build toward in section 5.4 — it means your "real" serving path (FastAPI + Docker + Vertex AI) stays exactly as-is, and the Streamlit app is just another client of it, same as any other consumer.

In [ ]:
!pip install -q --break-system-packages streamlit pandas scikit-learn joblib requests
print("done")

## 5.2 Project setup

Same idea as Module 3: we build a real project directory from inside the notebook with `%%writefile`, then run it with the actual `streamlit` CLI (not something you can drive through `TestClient` — Streamlit isn't a WSGI/ASGI app in the FastAPI sense, so we run it as a real process and hit it over HTTP, same as you'll do at the terminal).

```
streamlit_project/
├── app.py                    <- the data app (model loaded in-process)
├── app_api_backed.py         <- variant that calls a FastAPI backend instead
├── requirements.txt
├── Dockerfile
└── model_artifacts/
    └── model.joblib
```

In [ ]:
import os
os.makedirs("/home/claude/streamlit_project/model_artifacts", exist_ok=True)

import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=2000, n_features=3, n_informative=3,
                            n_redundant=0, random_state=42)
clf = LogisticRegression().fit(X, y)
joblib.dump(clf, "/home/claude/streamlit_project/model_artifacts/model.joblib")
print("model saved")

In [ ]:
%%writefile /home/claude/streamlit_project/requirements.txt
streamlit==1.62.0
pandas==2.2.3
scikit-learn==1.5.2
joblib==1.4.2
numpy==1.26.4
requests==2.32.3

## 5.3 The data app: widgets, single prediction, batch CSV scoring

A few things worth calling out as we build this:

- **`@st.cache_resource`** loads the model once per server process (not once per user interaction) — without it, every widget interaction would reload the pickle from disk, which is wasteful and, for a large model, slow enough to hurt the UX badly.
- **`st.tabs`** keeps single-row and batch scoring in one app without cluttering the layout.
- **`st.columns`** lays widgets out side by side instead of stacking everything vertically.
- Feature-contribution bar chart reuses the same "value × coefficient" idea from Module 2's exercise — a data app is a great place to make a model's reasoning visible to non-engineers.

In [ ]:
%%writefile /home/claude/streamlit_project/app.py
import joblib
import numpy as np
import pandas as pd
import streamlit as st

MODEL_PATH = "model_artifacts/model.joblib"
FEATURE_NAMES = ["feature_1", "feature_2", "feature_3"]

st.set_page_config(page_title="Fraud Model Explorer", page_icon="🔍", layout="centered")


@st.cache_resource
def load_model():
    return joblib.load(MODEL_PATH)


model = load_model()

st.title("🔍 Fraud Model Explorer")
st.caption("A data app for interacting with the fraud_v1_logreg model")

tab_single, tab_batch = st.tabs(["Single prediction", "Batch (CSV) prediction"])

with tab_single:
    st.subheader("Enter feature values")
    col1, col2, col3 = st.columns(3)
    with col1:
        feature_1 = st.slider("Account age (years)", 0.0, 10.0, 2.0, 0.1)
    with col2:
        feature_2 = st.number_input("Transactions (24h)", min_value=0, max_value=200, value=5)
    with col3:
        feature_3 = st.number_input("Avg transaction amount", min_value=0.0, max_value=100000.0, value=10.0)

    if st.button("Predict", type="primary"):
        X = np.array([[feature_1, feature_2, feature_3]])
        pred = int(model.predict(X)[0])
        proba = float(model.predict_proba(X)[0][pred])

        m1, m2 = st.columns(2)
        m1.metric("Prediction", "Fraud" if pred == 1 else "Legit")
        m2.metric("Confidence", f"{proba:.1%}")

        coefs = model.coef_[0]
        contributions = pd.DataFrame({
            "feature": FEATURE_NAMES,
            "value": [feature_1, feature_2, feature_3],
            "contribution": [feature_1 * coefs[0], feature_2 * coefs[1], feature_3 * coefs[2]],
        }).set_index("feature")
        st.bar_chart(contributions["contribution"])

with tab_batch:
    st.subheader("Upload a CSV for batch scoring")
    st.caption("Expected columns: feature_1, feature_2, feature_3")
    uploaded = st.file_uploader("Choose a CSV file", type="csv")

    if uploaded is not None:
        df = pd.read_csv(uploaded)
        missing = set(FEATURE_NAMES) - set(df.columns)
        if missing:
            st.error(f"Missing columns: {missing}")
        else:
            X = df[FEATURE_NAMES].to_numpy()
            preds = model.predict(X)
            probas = model.predict_proba(X)
            df["prediction"] = preds
            df["probability"] = [probas[i][p] for i, p in enumerate(preds)]

            st.dataframe(df, use_container_width=True)
            st.bar_chart(df["prediction"].value_counts())

            csv_bytes = df.to_csv(index=False).encode("utf-8")
            st.download_button("Download scored CSV", csv_bytes, "scored.csv", "text/csv")

### Run it

From a terminal, inside `streamlit_project/`:

```bash
streamlit run app.py
```

Streamlit opens `http://localhost:8501` in your browser automatically. Try the slider/number inputs, hit **Predict**, then switch to the batch tab and upload a CSV with `feature_1,feature_2,feature_3` columns.

You can also run it in the background from this notebook to sanity-check it boots without errors (useful in CI, or just to confirm nothing's broken before a live demo):

In [ ]:
%%bash
cd /home/claude/streamlit_project
nohup streamlit run app.py --server.headless true --server.port 8501 > /tmp/streamlit_app.log 2>&1 &
sleep 4
curl -s -o /dev/null -w "health check status: %{http_code}\n" http://localhost:8501/_stcore/health

In [ ]:
%%bash
# tear it down when you're done
pkill -f "streamlit run app.py" || true

## 5.4 Pattern 2 — Streamlit as a frontend for your FastAPI backend

Loading the model directly inside the Streamlit process (as above) is the fastest way to prototype, but it means **the app and the model are coupled** — every Streamlit replica needs its own copy of the model in memory, and you can't reuse the exact serving path you already built and tested in Modules 1–4.

The alternative: Streamlit becomes a **thin client** that calls your existing FastAPI `/predict` endpoint — the same one from Module 2, wrapped in Docker in Module 3, deployable to Vertex AI in Module 4. This is the more common production shape once a prototype graduates:

```
[ Streamlit app ]  --HTTP POST /predict-->  [ FastAPI service ]  -->  [ Model ]
   (UI only)              (with API key)         (your Module 1-4 stack)
```

The Streamlit code barely changes — swap the local `model.predict()` call for a `requests.post()` call, and reuse the API-key pattern from Module 1.

In [ ]:
%%writefile /home/claude/streamlit_project/app_api_backed.py
import requests
import streamlit as st

st.set_page_config(page_title="Fraud Model Explorer (API-backed)", page_icon="🔍")

st.title("🔍 Fraud Model Explorer")
st.caption("This version calls a FastAPI backend instead of loading the model in-process")

with st.sidebar:
    st.subheader("Backend settings")
    api_base_url = st.text_input("API base URL", value="http://localhost:8000")
    api_key = st.text_input("API key", value="supersecret-demo-key", type="password")

col1, col2, col3 = st.columns(3)
with col1:
    feature_1 = st.slider("Account age (years)", 0.0, 10.0, 2.0, 0.1)
with col2:
    feature_2 = st.number_input("Transactions (24h)", min_value=0, max_value=200, value=5)
with col3:
    feature_3 = st.number_input("Avg transaction amount", min_value=0.0, max_value=100000.0, value=10.0)

if st.button("Predict", type="primary"):
    payload = {"feature_1": feature_1, "feature_2": feature_2, "feature_3": feature_3}
    headers = {"X-API-Key": api_key}

    try:
        resp = requests.post(f"{api_base_url}/predict", json=payload, headers=headers, timeout=5)
    except requests.exceptions.ConnectionError:
        st.error(f"Could not reach the API at {api_base_url}. Is it running?")
        st.stop()

    if resp.status_code == 200:
        data = resp.json()
        m1, m2 = st.columns(2)
        m1.metric("Prediction", "Fraud" if data["prediction"] == 1 else "Legit")
        m2.metric("Confidence", f"{data['probability']:.1%}")
        st.caption(f"Served by model version: {data['model_version']}")
    elif resp.status_code == 401:
        st.error("API key was rejected — check the key in the sidebar.")
    else:
        st.error(f"API returned {resp.status_code}: {resp.text}")

### Try it against a real backend

1. In a separate terminal, run your Module 2/3 FastAPI service (with API-key auth from Module 1 layered on top) on port 8000.
2. In another terminal, run `streamlit run app_api_backed.py` and leave the default URL/key, or point it at wherever you deployed the service — including, eventually, a Cloud Run/GKE-fronted version of your Vertex AI-served model if you put a small routing layer in front of it.
3. Notice the app now handles three distinct failure modes explicitly: connection refused (backend not running), `401` (bad API key), and any other non-`200` (surfaced as a generic error) — this is the same defensive pattern from Module 2, just on the *calling* side this time instead of the serving side.

## 5.5 Dockerizing the Streamlit app

Nearly identical to Module 3's Dockerfile — the only real differences are the port (Streamlit's default is `8501`, not `8080`) and the health route (`/_stcore/health`, built into Streamlit).

In [ ]:
%%writefile /home/claude/streamlit_project/Dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app.py .
COPY model_artifacts/ ./model_artifacts/

RUN useradd --create-home appuser
USER appuser

ENV PORT=8501
EXPOSE 8501

HEALTHCHECK --interval=30s --timeout=3s --start-period=10s --retries=3 CMD python -c "import urllib.request, os; urllib.request.urlopen('http://localhost:' + os.environ.get('PORT', '8501') + '/_stcore/health')" || exit 1

CMD ["sh", "-c", "streamlit run app.py --server.port=${PORT} --server.address=0.0.0.0 --server.headless=true"]

In [ ]:
%%bash
cd /home/claude/streamlit_project
docker build -t fraud-data-app:latest .
docker run -d --name fraud-data-app -p 8501:8501 fraud-data-app:latest
sleep 3
curl -s -o /dev/null -w "%{http_code}\n" http://localhost:8501/_stcore/health

In [ ]:
%%bash
docker stop fraud-data-app
docker rm fraud-data-app

### Where would you actually deploy this?

Deliberately **not** Vertex AI Model/Endpoint — that resource type expects the `/predict`-with-`instances` prediction contract from Module 4, not a general-purpose web UI. For a Streamlit data app, the common managed targets are:
- **Cloud Run** — simplest option for a stateless containerized web app on GCP; scales to zero, HTTPS out of the box.
- **GKE** — if it needs to sit alongside other services you already run there.
- **Streamlit Community Cloud** — fine for public demos with no sensitive data.

The Docker image itself is portable across all three with no changes — that's the whole point of having containerized it in the first place.

## Exercise 5 (10–15 min)

1. Add a third tab to `app.py`: **model info** — display the model's coefficients and intercept in a table (`st.table` or `st.dataframe`), plus a short markdown description of what each feature means.
2. In `app_api_backed.py`, add a `st.spinner("Calling model...")` around the `requests.post` call, and add a simple client-side retry (2 attempts) if the request times out.
3. Bonus: add caching to the batch-scoring path using `@st.cache_data` keyed on the uploaded file's bytes, so re-running the app on the same file doesn't reprocess it — think about why `@st.cache_data` (not `@st.cache_resource`) is the right choice here (hint: it's about *serializable data* vs. *unserializable objects/connections*).

## Recap

- Streamlit turns a script into an app; it's for humans exploring a model or dataset, not for machine-to-machine production traffic — that's still FastAPI's job.
- `@st.cache_resource` for objects (models, DB connections); `@st.cache_data` for data (DataFrames, arrays) — using the wrong one either breaks caching or silently keeps stale mutable objects around.
- Two valid architectures: model in-process (fast to prototype) vs. calling your FastAPI backend (reuses your production serving path, keeps the UI and the model decoupled).
- Containerize it exactly like Module 3, just with Streamlit's port/health route — and deploy it somewhere meant for general web apps (Cloud Run/GKE), not as a Vertex AI Model/Endpoint.